In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py UpdraftArea

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
# #IN JUPYTER LAB (disable otherwise)
dataName='Variables'
# dataName='UpdraftArea'
# dataName='Entrainment'
# dataName='ResidenceTime'

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm
# from scipy.interpolate import interp1d  
from scipy import stats
from matplotlib.ticker import LogLocator
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
#Get Data
# def GetData_V1(trackedArray):
#     pIndices=trackedArray[:,0]
    
#     varNames=GetVarNames()
    
#     dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
#     for t in tqdm(range(ModelData.Ntime), position=2, leave=False):
#         for varName in varNames:
#             if any(prefix in var for var in varNames for prefix in ("E_")) and t == ModelData.Ntime-1:
#                 dataDictionary[varName][t] = np.full((1, len(pIndices)), np.nan)
#             else:
#                 ###############
#                 dataDictionary[varName][t] = CallLagrangianArray(ModelData, DataManager, \
#                                                                  ModelData.timeStrings[t], varName)[pIndices]
#                 ###############

#     return dataDictionary

def GetData_V2(trackedArray):
    pIndices=trackedArray[:,0]
    
    varNames=GetVarNames()
    dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
    for t in tqdm(range(ModelData.Ntime), position=2, leave=False):
        dataLocationInfoCache = {'inputDataDirectory': None, 'dataName': None}; currentFile = None
        
        for varName in varNames:
            if ("E_" in varName or "D_" in varName) and (t == ModelData.Ntime-1):
                dataDictionary[varName][t] = np.full((1, len(pIndices)), np.nan)
            else:
                ###############
                [inputDataDirectory,dataName] = CallLagrangianArray(ModelData, DataManager, \
                                                                    ModelData.timeStrings[t], \
                                                                    varName,loadData=False) #get file info
                if inputDataDirectory != dataLocationInfoCache['inputDataDirectory']\
                or dataName != dataLocationInfoCache['dataName']: #if not in cache, open new file
                    # print('reloading',varName)
                    if currentFile is not None: currentFile.close() #close the old file
                    currentFile=DataManager.GetTimestepData_V2(inputDataDirectory, ModelData.timeStrings[t], dataName)
                    dataLocationInfoCache['inputDataDirectory']=inputDataDirectory #set cache
                    dataLocationInfoCache['dataName']=dataName #set cache
                dataDictionary[varName][t] = currentFile[varName][:][pIndices] #grab the needed variable
                ###############
        if currentFile is not None: currentFile.close() #close the file at end of each timestep
    return dataDictionary

#Entrainment Variable Things
def GetVarNames():
    varNames = ['z','Z','LFC','QCQI','W']#,'A_g','A_c']
    if dataName == 'Variables':
        varNames += ['VMF_g','VMF_c','BUOYANCY2']
        varNames += ['QV','QCQI','RH_vapor','HMC']
        varNames += ['THETA_v','THETA_e']
        varNames += ['X']
        
    elif dataName == "Entrainment":
        PROCESSED_string = "PROCESSED_" #if "test" not in dataName else "" #*testing
        varNames += ['Z']
        for updraftType in ['g','c']:
            for DivideMassFlux_string in ["","_DivideMassFlux"]:#,"_DivideMassFluxMean"]:
                varNames += [f'{PROCESSED_string}E{DivideMassFlux_string}_{updraftType}',f'{PROCESSED_string}D{DivideMassFlux_string}_{updraftType}']
    elif dataName == "Eulerian_Entrainment":
        for updraftType in ['g','c']:
            for DivideMassFlux_string in ["","_DivideMassFlux"]:#,"_DivideMassFluxMean"]:
                varNames += [f'E_{updraftType}_eulerian{DivideMassFlux_string}',f'D_{updraftType}_eulerian{DivideMassFlux_string}']
                
    elif "UpdraftArea" in dataName:
        for updraftType in ['g','c']:
            varNames += [f'UPDRAFT_AREA_{updraftType}',f'UPDRAFT_EDGE_DISTANCE_{updraftType}']
            
    elif "ResidenceTime" in dataName:   
        varNames += ['Y','X']
        
    return varNames

def GetMeanVarNames():
    varNames=[]
    if dataName == 'Variables':
        varNames+=['QV','RH_vapor','HMC']
        varNames+=['THETA_v','THETA_e']
    return varNames

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
def RunCode(trackedArrays,parcelTypes):

    parcelDepths=['ALL']
    
    #subset out parcelType and parcelDepth
    trajectoriesArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    priorToAscentArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    for parcelType in tqdm(parcelTypes, position=0, leave=True):
        for parcelDepth in tqdm(parcelDepths, desc=f"Type: {parcelType}", position=1, leave=False):

            #subsetting data
            trackedArray = trackedArrays[parcelType][parcelDepth]
            t1 = trackedArray[:, 1].astype(int); t2 = trackedArray[:, 2].astype(int)
            after = trackedArray[:, 3].astype(int)
        
            # Data Calculations
            # loading variable data dictionary
            tqdm.write(f"Getting Data")
            dataDictionary = GetData_V2(trackedArray)
            
            #getting data
            varNames=GetVarNames()
            for varName in tqdm(varNames, desc="Subsetting Variable", leave=False):
                variableData = dataDictionary[varName]
                
                #computing trajectories for each variable
                trajectoriesArray = SubsetTrajectories(variableData,t1,t2,after)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = trajectoriesArray
                
                priorToAscentArray = SubsetTrajectories(variableData,t1=0,t2=t1,after=0)
                priorToAscentArrayDictionary[parcelType][parcelDepth][varName] = priorToAscentArray

    return trajectoriesArrayDictionary,priorToAscentArrayDictionary

def SubsetTrajectories(variableData,t1,t2,after):
    #making output array
    trajectoriesArray = np.full_like(variableData,fill_value=np.nan,dtype=float)

    #masking the final output array
    time_idx = np.arange(variableData.shape[0])[:, np.newaxis]
    mask = (time_idx >= t1) & (time_idx <= t2+after)
    
    #applying mask to output array
    trajectoriesArray[mask] = variableData[mask]

    return trajectoriesArray

def LimitTrackedArraysRows(trackedArrays, limit=None): #limit=(0,10000)
    if limit is None:
        return trackedArrays
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            trackedArrays[parcelType][parcelDepth] \
            = trackedArrays[parcelType][parcelDepth][limit[0]:limit[1], :]
    return trackedArrays
    
# #estimating file size
# Np=107076; Nt=661; Nvars=5;
# Nparceltypes=2
# Np*Nt*Nvars*Nparceltypes*4/1e9

In [ ]:
##############################################
#COMPUTING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
running=False

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
    # trackedArrays=LimitTrackedArraysRows(trackedArrays) #limiting number of parcels to allow computation to complete

    parcelTypesList = [['CL','nonCL']]
    for parcelTypes in parcelTypesList:
        [trajectoriesArrayDictionary,priorToAscentArrayDictionary] = RunCode(trackedArrays,parcelTypes)
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles,trajectoriesArrayDictionary, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles,priorToAscentArrayDictionary, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')

In [ ]:
##############################################
#DATA LOADING AND POST-PROCESSING FUNCTIONS

In [ ]:
#Loading SBF X Locations

def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence
    
def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs

if not running:
    SBF_XLocations = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager)

In [ ]:
#Data Loading and Post-Processing Functions
def LoadAndProcessData(subsetting_RightOfSBF=False,subsetting_time=None,dataName=dataName):

    parcelTypesList = [['CL','nonCL']]; parcelTypes = parcelTypesList[0]
    trajectoriesArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
    priorToAscentArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')
    SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary,dataName)
    RemoveLastValidTimestep(trajectoriesArrayDictionary)  # trim CU exit timestep before subsetting
    if dataName == 'Variables':
        SubtractSBF_XLocation(trajectoriesArrayDictionary,priorToAscentArrayDictionary,SBF_XLocations)
        SetZeroMassFluxToNaN(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    if dataName == 'Entrainment': #or dataName == 'Entrainment_test': #*testing
        FixDetrainmentSign(trajectoriesArrayDictionary); FixDetrainmentSign(priorToAscentArrayDictionary)
        CalculateNetEntrainment(trajectoriesArrayDictionary); CalculateNetEntrainment(priorToAscentArrayDictionary)
    if dataName == 'Eulerian_Entrainment':
        CalculateNetEntrainment(trajectoriesArrayDictionary); CalculateNetEntrainment(priorToAscentArrayDictionary)

    if dataName == 'ResidenceTime':
        AddTimeRelativeToInitialBLAscent(trajectoriesArrayDictionary)

    #subsetting only right of SBF
    if subsetting_RightOfSBF:
        trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
        LimitLeftOrRightOfSBF(trajectoriesArrayDictionary,trackedArrays,leftRight='right')
        LimitLeftOrRightOfSBF(priorToAscentArrayDictionary,trackedArrays,leftRight='right')
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting time range
    if subsetting_time is not None:
        SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,subsetting_time)
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting supersets
    for arrayDictionary in [trajectoriesArrayDictionary,priorToAscentArrayDictionary]:
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='SBF')
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='ColdPool')
        
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def LimitLeftOrRightOfSBF(dictionary,trackedArrays,
                          leftRight='right'):
    leftRightIdentifier=1 if leftRight=='right' else -1
    
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            leftRightList = trackedArrays[parcelType][parcelDepth][:,4] 
            mask = leftRightList == leftRightIdentifier
            for varName in dictionary[parcelType][parcelDepth]:
                dictionary[parcelType][parcelDepth][varName]=dictionary[parcelType][parcelDepth][varName][:,mask]
    # return dictionary
    
def SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,
               subsetting_time):
    times=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    mask = (times >= (subsetting_time[0] or -np.inf)) & (times <= (subsetting_time[1] or np.inf))
    timeIndices=np.arange(ModelData.Ntime)[mask]
    
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                first_valid = np.argmax(~np.isnan(data_1), axis=0)
                keep_mask = np.isin(first_valid, timeIndices)
                data_1[:, ~keep_mask] = np.nan
                data_2[:, ~keep_mask] = np.nan
                
    # return trajectoriesArrayDictionary,priorToAscentArrayDictionary

def RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary):
    """
    use after applying SubsetTime()
    """
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                keep_mask = np.any(~np.isnan(data_1), axis=0)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = data_1[:, keep_mask]
                priorToAscentArrayDictionary[parcelType][parcelDepth][varName] = data_2[:, keep_mask]
                
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary,dataName_Global):

    if dataName_Global != 'Variables': return
        
    DataManager_CalculateMeans = DataManager_Class(mainDirectory, scratchDirectory, ModelData, 
                                                   dataType="CalculateMoreVariables",dataName="CalculateMeans",dtype='float32',
                                                   verbose=False)
    
    [inputDataDirectory,dataName] = CallVariable(ModelData, DataManager_CalculateMeans, 'all_times', 'qv_mean',loadData=False) 
    meanData = DataManager.GetTimestepData_V2(inputDataDirectory, 'all_times', dataName=dataName)
    
    varNames = GetMeanVarNames()
    for varName in varNames:
        varNameMean = varName.lower()+'_mean'
        if varNameMean not in meanData: continue        
        varMean = meanData[varNameMean][:]  # shape (t, Nz)
        for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
            for parcelType in dictionary:
                for parcelDepth in dictionary[parcelType]:
                    d = dictionary[parcelType][parcelDepth]
                    zIndexes = d['Z']
                    validMask = ~np.isnan(zIndexes)
    
                    perturbation = np.full(d[varName].shape, np.nan)
                    tIndexes = np.arange(varMean.shape[0])[:, np.newaxis] * np.ones_like(zIndexes, dtype=int)
                    
                    perturbation[validMask] = d[varName][validMask] - varMean[tIndexes[validMask], zIndexes[validMask].astype(int)]    
                    dictionary[parcelType][parcelDepth][varName+'_perturbation'] = perturbation
                    
    #close data
    meanData.close()

def RemoveLastValidTimestep(trajectoriesArrayDictionary):
    """
    Removes the last non-nan value from each parcel across all variables.
    Sets the last valid timestep to NaN for every parcel and variable.
    """
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            d = trajectoriesArrayDictionary[parcelType][parcelDepth]
            
            # Use 'z' as reference to find last valid timestep per parcel
            z = d['z']
            validMask = np.isfinite(z)  # shape (t, p)
            
            # For each parcel, find the index of the last valid timestep
            # argmax on flipped axis gives first True from the end
            last_valid_idx = z.shape[0] - 1 - np.argmax(validMask[::-1, :], axis=0)  # shape (p,)
            ever_valid = validMask.any(axis=0)  # guard against all-nan parcels
            
            # Set last valid timestep to NaN for all variables
            for varName in d:
                for p in np.where(ever_valid)[0]:
                    d[varName][last_valid_idx[p], p] = np.nan

def SubtractSBF_XLocation(trajectoriesArrayDictionary, priorToAscentArrayDictionary, SBF_XLocations):

    SBF_XLocations = np.asarray(SBF_XLocations, dtype=float)

    for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
        for parcelType in dictionary:
            for parcelDepth in dictionary[parcelType]:

                d = dictionary[parcelType][parcelDepth]

                xData = d['X']
                validMask = ~np.isnan(xData)

                tIndexes = np.arange(xData.shape[0])[:, np.newaxis] * np.ones_like(xData, dtype=int)

                sbfForParcels = SBF_XLocations[tIndexes]

                validMask = validMask & ~np.isnan(sbfForParcels)

                xPerturbation = np.full(xData.shape, np.nan)
                xPerturbation[validMask] = xData[validMask] - sbfForParcels[validMask]

                d['X_SBF_Relative'] = xPerturbation

def SetZeroMassFluxToNaN(trajectoriesArrayDictionary,priorToAscentArrayDictionary):
    """
    Replace zeros with NaN for VMF_g and VMF_c variables.
    """
    for dictionary in [trajectoriesArrayDictionary,priorToAscentArrayDictionary]:
        for parcelType in dictionary:
            for parcelDepth in dictionary[parcelType]:
                variables = dictionary[parcelType][parcelDepth]
                for varName in ['VMF_g', 'VMF_c']:
                    if varName in variables:
                        variables[varName] = np.where(variables[varName] == 0,
                                                      np.nan,variables[varName])

def FixDetrainmentSign(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            for varName in dictionary[parcelType][parcelDepth]:
                PROCESSED_string = 'PROCESSED_' if 'PROCESSED_' in varName else ""
                DivideMassFlux_string = '_DivideMassFlux' if '_DivideMassFlux' in varName else ""
                if f"{PROCESSED_string}D{DivideMassFlux_string}_" in varName:
                    dictionary[parcelType][parcelDepth][varName] *= -1

def CalculateNetEntrainment(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            variables = dictionary[parcelType][parcelDepth]
            varNames = list(variables.keys())
            entrainment_varNames = [varName for varName in varNames if "E_" in varName]
            for entrainment_varName in entrainment_varNames:
                detrainment_varName = entrainment_varName.replace('E_', 'D_')
                net_varName = entrainment_varName.replace('E_', 'NET_')
                variables[net_varName] = variables[entrainment_varName] - variables[detrainment_varName]

def AddTimeRelativeToInitialBLAscent(trajectoriesArrayDictionary):
    
    time = (np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6) * 60
    
    for parcelType in trajectoriesArrayDictionary.keys():
        for parcelDepth in trajectoriesArrayDictionary[parcelType].keys():
            dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth]
            
            Z = dataDictionary['Z']
            z = dataDictionary['z']
            W = dataDictionary['W']
            qcqi = dataDictionary['QCQI']
            
            masks = {
                'ascent': np.isfinite(Z) & (W > 0.1),
                'CB':     np.isfinite(Z) & (qcqi > 1e-6),
                'CU':     np.isfinite(Z) & (W > 0.5) & (qcqi > 1e-6),
            }
            
            for suffix, mask in masks.items():
                first_idx = np.argmax(mask, axis=0)
                
                # time_at_first = time[first_idx]
                never_met = ~np.any(mask, axis=0) #just in case all nans, skips those cases
                time_at_first = np.where(never_met, np.nan, time[first_idx]) #just in case all nans, skips those cases
                
                time_since = time[:, None] - time_at_first[None, :]
                time_since = np.where(mask, time_since, np.nan)

                dataDictionary[f'z_since_{suffix}'] = np.where(mask, z, np.nan)
                dataDictionary[f'W_since_{suffix}'] = np.where(mask, W, np.nan)
                dataDictionary[f'time_since_{suffix}'] = time_since
                # AddNormalizedTimeSinceAscent(dataDictionary, key, mask) #depreciated

    return trajectoriesArrayDictionary
    

# def AddNormalizedTimeSinceAscent(dataDictionary, timeKey, mask): #depreciated

#     T = dataDictionary[timeKey]          # min
#     zHeight = dataDictionary['z'] / 1e3  # km

#     zMasked = np.where(mask, zHeight, np.nan)
#     zStart = np.nanmin(zMasked, axis=0)

#     dzFromStart = zMasked - zStart[None, :]

#     with np.errstate(invalid='ignore', divide='ignore'):
#         normalizedAge = T / dzFromStart

#     dataDictionary[f'{timeKey}_per_km'] = normalizedAge

def SubsetDataParcelType(arrayDictionary,parcelType_original='CL', parcelType_subset='SBF'):
    """
    In arrayDictionary, the parcelType CL is a superset of SBF and ColdPool.
    This function allows simple "isin" mask to create new parcelTypes for the subset of each superset,
    using indicies marking each subset within trackedArrays (loaded below), as calculated from the SubsetParcels code.
    """

    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class,verbose=False)
    
    original = trackedArrays[parcelType_original]['ALL']
    subset = trackedArrays[parcelType_subset]['ALL']
    
    mask = np.isin(original[:,0], subset[:,0])
    
    arrayDictionary[parcelType_subset] = {}
    
    for inner_key1 in arrayDictionary[parcelType_original]:
        arrayDictionary[parcelType_subset][inner_key1] = {}
        for inner_key2 in arrayDictionary[parcelType_original][inner_key1]:
            varData = arrayDictionary[parcelType_original][inner_key1][inner_key2]
            arrayDictionary[parcelType_subset][inner_key1][inner_key2] = varData[:, mask]
    
    return arrayDictionary

In [ ]:
#Loading Data
if not running:
    subsetting_RightOfSBF=False; #subsetting_RightOfSBF=True
    subsetting_time=None; #subsetting_time=(12,None)
    
    [trajectoriesArrayDictionary,priorToAscentArrayDictionary]\
    =LoadAndProcessData(subsetting_RightOfSBF=subsetting_RightOfSBF,subsetting_time=subsetting_time)

In [ ]:
##############################################
#Analysis

In [ ]:
##############################################
#HISTOGRAM PLOTTING FUNCTIONS

In [ ]:
#Plotting Functions
def PlotBinnedData(meanData_1, meanData_2, time_bins, z_bins,
                   varName,
                   # interpolate=False, 
                   multiplier=1,units='',
                   cmap='RdBu_r', symmetric=True,
                   label='',
                   zLabel='z - z_LFC (m)',
                   titles=['Panel 1', 'Panel 2'],
                   xLabel='time since right before ascent (mins)',
                   plotType='contourf',numLevels=21,
                   axes=None):

    #set figure and axis
    if axes is None:
        fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True, sharey=True)
    # else:
    #     fig = axes[0].get_figure() #could use later to make a function that combines all plots

    #set background color
    if plotType == 'contourf':
        for axis in axes.ravel():
            axis.set_facecolor('grey')

    #get vmin_threshold
    vmin_threshold = 1e-6 if varName == "QCQI" else None
    
    #organize data
    meanDatas = [meanData_1, meanData_2]
    #apply multiplier
    meanDatas = [multiplier * meanData for meanData in meanDatas]
    if vmin_threshold is not None: vmin_threshold*=multiplier

    # #interpolate
    # if interpolate: meanDatas = [InterpolateBinnedData(meanData,time_bins) for meanData in meanDatas]

    #Coordinates
    if plotType == 'pcolormesh':
        plotX = time_bins[:-1]; plotY = z_bins[:-1]
    elif plotType == 'contourf':
        plotX = 0.5 * (time_bins[:-1] + time_bins[1:]); plotY = 0.5 * (z_bins[:-1] + z_bins[1:])

    #single plots colorbar setup
    if vmin_threshold is not None:
        meanDatas = [np.where(meanData >= vmin_threshold, meanData, np.nan)
                     for meanData in meanDatas]
    if symmetric:
        extend = 'both'
        vmax = np.nanmax([np.nanpercentile(np.abs(meanData), 95)
                          for meanData in meanDatas])
        vmin = vmin_threshold if vmin_threshold is not None else -vmax
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        levels=np.linspace(vmin,vmax,numLevels)
    else:
        extend = 'max' if varName=='QCQI' else 'both'
        vmin = vmin_threshold if vmin_threshold is not None else \
        np.nanmin([np.nanpercentile(meanData, 5)for meanData in meanDatas])
        vmax = np.nanmax([np.nanpercentile(meanData, 95) for meanData in meanDatas])
        if vmax == 0 and '_E_' in varName or '_D_' in varName:
            vmax = np.nanmax([np.nanmax(meanData) for meanData in meanDatas])
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        levels=np.linspace(vmin,vmax,numLevels)

    #difference colorbar setup
    differenceData = meanDatas[0]-meanDatas[1]
    diff_vmax = np.nanpercentile(np.abs(differenceData), 95)
    diff_vmin = -diff_vmax
    diff_norm = mcolors.TwoSlopeNorm(vmin=diff_vmin, vcenter=0, vmax=diff_vmax)
    diff_levels = np.linspace(diff_vmin, diff_vmax, numLevels)
        
    #Single Plots
    for axis, meanData, title in zip(axes, meanDatas, titles):

        if plotType == 'pcolormesh':
            im = axis.pcolormesh(plotX,plotY,meanData.T,
                                 cmap=cmap,norm=norm)
        elif plotType == 'contourf':
            im = axis.contourf(plotX,plotY,meanData.T,
                               levels=levels,cmap=cmap,norm=norm,extend=extend)

        axis.axhline(0, color='grey', linestyle='--')
        axis.set_ylabel(zLabel); axis.set_title(title)


    #Difference Plot
    if plotType == 'pcolormesh':
        im_diff = axes[2].pcolormesh(plotX,plotY,differenceData.T,
                                     cmap='RdBu_r',norm=diff_norm)
    elif plotType == 'contourf':
        im_diff = axes[2].contourf(plotX,plotY,differenceData.T,
                                   levels=diff_levels,cmap='RdBu_r',norm=diff_norm,extend='both')
        
    axes[2].axhline(0, color='grey', linestyle='--')
    axes[2].set_ylabel(zLabel); axes[2].set_title(f'{titles[0]} - {titles[1]}')
    axes[-1].set_xlabel(xLabel)

    #set colorbars
    fig.colorbar(im,ax=axes[:2],label=fr'${label}$ ({units})',extend=extend)
    fig.colorbar(im_diff,ax=axes[2],label=fr'$\Delta {label}$ ({units})',extend='both')

    #set axis limits
    for axis in axes.ravel():
        axis.set_ylim(z_bins[0],z_bins[-1])
    return fig

def GetPlottingDirectory(plotFileName, parcelTypes,parcelDepth, extension='pdf'):
    def GetPlotType(dataName):
        plotType=f"Project_Algorithms/Tracked_Profiles/Tracked_Ascent_Trajectories_{dataName}"
        return plotType
    def GetFolderName(parcelTypes,parcelDepth):
        leftRightString = 'RightOfSBF' if subsetting_RightOfSBF is True else 'everywhere'
        folderName=f"{leftRightString.upper()}/{parcelTypes[0]}_vs_{parcelTypes[1]}/{parcelDepth}/"
        return folderName
        
    plotType = GetPlotType(dataName)
    folderName = GetFolderName(parcelTypes,parcelDepth)

    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    simStr = f"{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz"
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, f"Simulation_{simStr}", folderName)
    os.makedirs(specificPlottingDirectory, exist_ok=True)
    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName+f'.pdf')

    print(f'Plotting Location: {f'Saving to: {plottingFileName}'}')
    return plottingFileName
def SaveFigureToPDF(pdf,fig,plottingFileName):
    pdf.savefig(fig); plt.close(fig)

In [ ]:
#Label Dictionaries
def GetMultiplierDictionary():
    #Variables
    multiplierDictionary = {'QV':        1e3,'RH_vapor':  1e2,'QCQI':      1e3,
                            'THETA_v':   1,'THETA_e':   1,
                            'W':         1,'VMF_g':     1,'VMF_c':     1,
                            'HMC':       1e3,'BUOYANCY2': 1,
                            'z':         1/1e3, 'X': ModelData.kms, 'X_SBF_Relative': ModelData.kms}

    perturbationEntries = {k + '_perturbation': v for k, v in multiplierDictionary.items() if k in GetMeanVarNames()}
    multiplierDictionary.update(perturbationEntries)

    #Entrainment
    PROCESSED_string = "PROCESSED_" #if "test" not in dataName else "" #*testing
    multiplierDictionary.update({
        f'{PROCESSED_string}{prefix}{DivideMassFlux_string}_{updraftType}': 1e3\
        if DivideMassFlux_string == '_DivideMassFlux' else 1
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ['', '_DivideMassFlux']
        for prefix in ['E', 'D', 'NET']
    })
    #Eulerian Entrainment
    multiplierDictionary.update({
        f'{prefix}_{updraftType}_eulerian{DivideMassFlux_string}': 1e3\
        if DivideMassFlux_string == '_DivideMassFlux' else 1
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ['', '_DivideMassFlux']
        for prefix in ['E', 'D', 'NET']
    })

    #UpdraftArea
    multiplierDictionary.update({
        f'{variable}_{updraftType}': multiplier
        for updraftType in ['g', 'c']
        for variable, multiplier in zip(
            ['UPDRAFT_AREA', 'UPDRAFT_EDGE_DISTANCE'],
            [1e-6, 1e-3]
        )
    })
        
    return multiplierDictionary
multiplierDictionary=GetMultiplierDictionary()

def GetUnitsDictionary():
    #Variables
    unitsDictionary = {'QV':        r'$g\ kg^{-1}$','RH_vapor':  r'$\%$', 'QCQI':      r'$g\ kg^{-1}$',
                       'THETA_v':   r'$K$','THETA_e':   r'$K$',
                       'W':         r'$m\ s^{-1}$','VMF_g':     r'$kg\ m^{-2}\ s^{-1}$','VMF_c':     r'$kg\ m^{-2}\ s^{-1}$',
                       'HMC':       r'$g\ kg^{-1}\ s^{-1}$','BUOYANCY2': r'$m\ s^{-2}$',
                       'z':         r'$km$', 'X': r'$km$', 'X_SBF_Relative': r'$km$'}
    perturbationEntries = {k + '_perturbation': v for k, v in unitsDictionary.items() if k in GetMeanVarNames()}
    unitsDictionary.update(perturbationEntries)
    
    #Entrainment
    PROCESSED_string = "PROCESSED_" #if "test" not in dataName else "" #*testing
    unitsDictionary.update({
            f'{PROCESSED_string}{prefix}{DivideMassFlux_string}_{updraftType}': r'$km^{-1}$' 
        if DivideMassFlux_string == '_DivideMassFlux' else r'$kg\ m^{-3}\ s^{-1}$'
            for updraftType in ['g', 'c']
            for DivideMassFlux_string in ['', '_DivideMassFlux']
            for prefix in ['E', 'D', 'NET']
        })
    #Eulerian Entrainment
    unitsDictionary.update({
            f'{prefix}_{updraftType}_eulerian{DivideMassFlux_string}': r'$km^{-1}$' 
        if DivideMassFlux_string == '_DivideMassFlux' else r'$kg\ m^{-3}\ s^{-1}$'
            for updraftType in ['g', 'c']
            for DivideMassFlux_string in ['', '_DivideMassFlux']
            for prefix in ['E', 'D', 'NET']
        })
    
    #UpdraftArea
    unitsDictionary.update({
        f'{variable}_{updraftType}':
            r'$km^{2}$' if variable == 'UPDRAFT_AREA'
            else r'$km$'
        for updraftType in ['g', 'c']
        for variable in ['UPDRAFT_AREA', 'UPDRAFT_EDGE_DISTANCE']
    })

    return unitsDictionary
unitsDictionary=GetUnitsDictionary()

def GetCmapDictionary():
    #Variables
    cmapDictionary = {'QV':        'turbo','RH_vapor':  'turbo','QCQI':      'turbo',
                      'THETA_v':   'turbo','THETA_e':   'turbo',
                      'W':         'RdBu_r','VMF_g':     'RdBu_r','VMF_c':     'RdBu_r',
                      'HMC':       'RdBu_r','BUOYANCY2': 'RdBu_r', 'z':         'viridis'}
                      

    perturbationEntries = {k + '_perturbation': 'RdBu_r' for k in cmapDictionary if k in GetMeanVarNames()}
    cmapDictionary.update(perturbationEntries)

    #Entrainment
    PROCESSED_string = "PROCESSED_" #if "test" not in dataName else "" #*testing
    cmapDictionary.update({
        f'{PROCESSED_string}{prefix}{DivideMassFlux_string}_{updraftType}': color
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ["", "_DivideMassFlux"]
        for prefix, color in zip(['E', 'D', 'NET'], ['turbo', 'turbo', 'RdBu_r'])
    })
    #Eulerian Entrainment
    cmapDictionary.update({
        f'{prefix}_{updraftType}_eulerian{DivideMassFlux_string}': color
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ["", "_DivideMassFlux"]
        for prefix, color in zip(['E', 'D', 'NET'], ['turbo', 'turbo', 'RdBu_r'])
    })

    #UpdraftArea
    cmapDictionary.update({
        f'{variable}_{updraftType}': 'turbo'
        for updraftType in ['g', 'c']
        for variable in ['UPDRAFT_AREA', 'UPDRAFT_EDGE_DISTANCE']
    })
    
    return cmapDictionary
cmapDictionary=GetCmapDictionary()

def GetLabelDictionary():

    # Base variable labels
    labelDictionary = {'QV': r'q_v', 'RH_vapor': r'RH_v', 'QCQI': r'q_c+q_i',
                       'THETA_v': r'\theta_v', 'THETA_e': r'\theta_e',
                       'W': r'w', 'VMF_g': r'VMF_g', 'VMF_c': r'VMF_c',
                       'HMC': r'HMC', 'BUOYANCY2': r'B', 'z': r'z'}

    # Perturbations
    labelDictionary.update({
        f'{var}_perturbation': rf'{{{label}}}^\prime'
        for var, label in labelDictionary.items()
        if var in GetMeanVarNames()
    })

    # Entrainment
    PROCESSED_string = "PROCESSED_"
    labelDictionary.update({
        f'{PROCESSED_string}{prefix}{DivideMassFlux_string}_{updraftType}':
            (
                rf'(e-d)_{{{updraftType}}}\,/\,M_{{{updraftType}}}'
                if prefix == 'NET' and DivideMassFlux_string == '_DivideMassFlux'
                else rf'{prefix.lower()}_{{{updraftType}}}\,/\,M_{{{updraftType}}}'
                if DivideMassFlux_string == '_DivideMassFlux'
                else rf'(e-d)_{{{updraftType}}}'
                if prefix == 'NET'
                else rf'{prefix.lower()}_{{{updraftType}}}'
            )
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ["", "_DivideMassFlux"]
        for prefix in ['E', 'D', 'NET']
    })
    
    # Eulerian entrainment
    labelDictionary.update({
        f'{prefix}_{updraftType}_eulerian{DivideMassFlux_string}':
            (
                rf'(e-d)_{{{updraftType}}}\,/\,M_{{{updraftType}}}'
                if prefix == 'NET' and DivideMassFlux_string == '_DivideMassFlux'
                else rf'{prefix.lower()}_{{{updraftType}}}\,/\,M_{{{updraftType}}}'
                if DivideMassFlux_string == '_DivideMassFlux'
                else rf'(e-d)_{{{updraftType}}}'
                if prefix == 'NET'
                else rf'{prefix.lower()}_{{{updraftType}}}'
            )
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ["", "_DivideMassFlux"]
        for prefix in ['E', 'D', 'NET']
    })
            
    # Updraft properties
    labelDictionary.update({
        f'{variable}_{updraftType}': label
        for updraftType in ['g', 'c']
        for variable, label in (
            ('UPDRAFT_AREA', rf'\mathrm{{Area}}_{{{updraftType}}}'),
            ('UPDRAFT_EDGE_DISTANCE', rf'\mathrm{{EdgeDistance}}_{{{updraftType}}}')
        )
    })

    return labelDictionary
# def PlotLabelDictionary(labelDictionary, ncols=4):

#     import math

#     n = len(labelDictionary)
#     nrows = math.ceil(n / ncols)

#     fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 1.8*nrows))
#     axes = np.atleast_1d(axes).ravel()

#     for ax, (varName, label) in zip(axes, labelDictionary.items()):
#         ax.set_xlim(0, 1)
#         ax.set_ylim(0, 1)

#         ax.set_xticks([0.5])
#         ax.set_xticklabels([rf'${label}$'], fontsize=14)

#         ax.set_yticks([])
#         ax.tick_params(axis='x', length=0)

#         ax.set_title(varName, fontsize=10)

#         # for spine in ax.spines.values():
#         #     spine.set_visible(False)

#     # Hide unused axes
#     for ax in axes[n:]:
#         ax.axis('off')

#     plt.tight_layout()
labelDictionary=GetLabelDictionary()

def isVariableSymmetric(varName):
    #Variables
    binaryDictionary = {'QV':         False,'RH_vapor':   False,'QCQI':       False,
                        'THETA_v':    False,'THETA_e':    False,
                        'W':          True,'VMF_g':      True,'VMF_c':      True,
                        'HMC':        True,'BUOYANCY2':  True,
                        'z':          False, 'X': False, 'X_SBF_Relative': True}

    perturbationEntries = {k + '_perturbation': True for k in cmapDictionary if k in GetMeanVarNames()}
    binaryDictionary.update(perturbationEntries)

    #Entrainment
    PROCESSED_string = "PROCESSED_" #if "test" not in dataName else "" #*testing
    binaryDictionary.update({
        f'{PROCESSED_string}{prefix}{DivideMassFlux_string}_{updraftType}': prefix == 'NET'
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ['', '_DivideMassFlux']
        for prefix in ['E', 'D', 'NET']
    })
    #Eulerian Entrainment
    binaryDictionary.update({
        f'{prefix}_{updraftType}_eulerian{DivideMassFlux_string}': prefix == 'NET'
        for updraftType in ['g', 'c']
        for DivideMassFlux_string in ['', '_DivideMassFlux']
        for prefix in ['E', 'D', 'NET']
    })

    #UpdraftArea
    binaryDictionary.update({
        f'{variable}_{updraftType}': value
        for updraftType in ['g', 'c']
        for variable, value in zip(
            ['UPDRAFT_AREA', 'UPDRAFT_EDGE_DISTANCE'],
            [False, False]
        )
    })
    
    return binaryDictionary.get(varName)

def GetPlottingVarNames():
    if dataName == 'Variables':
        varNames=[]
        varNames+=['W','VMF_g','VMF_c']
        varNames+=['QV','QCQI','RH_vapor','HMC']
        varNames+=['THETA_v','THETA_e','BUOYANCY2']
        
        varNames+=[meanVarName+'_perturbation' for meanVarName in GetMeanVarNames()] #adding perturbations
        varNames.remove('HMC_perturbation')
        
    elif dataName == "Entrainment":
        PROCESSED_string = "PROCESSED_" #if "test" not in dataName else "" #*testing
        varNames = [f'{PROCESSED_string}{suffix}{DivideMassFlux_string}_{updraftType}'\
                    for updraftType in ['g', 'c'] for suffix in ['E', 'D','NET']\
                    for DivideMassFlux_string in ["","_DivideMassFlux"]]
        
    elif dataName == "Eulerian_Entrainment":
        varNames = [f'{suffix}_{updraftType}_eulerian{DivideMassFlux_string}'\
                    for updraftType in ['g', 'c'] for suffix in ['E', 'D','NET']\
                    for DivideMassFlux_string in ["","_DivideMassFlux"]]
        
    elif "UpdraftArea" in dataName:
        varNames = [
            f'{variable}_{updraftType}'
            for updraftType in ['g', 'c']
            for variable in ['UPDRAFT_AREA', 'UPDRAFT_EDGE_DISTANCE']
        ]
    return varNames

In [ ]:
##############################################
#HISTOGRAM CALCULATION FUNCTIONS

In [ ]:
#Calculation Functions
def BinData(PrepareData_Function,
            dataDictionary,varName='W',relative='LFC',ascentTimeRelative=True):    
    _,_,_, time_bins,z_bins = PrepareData_Function(dataDictionary, 0, varName=varName,relative=relative,ascentTimeRelative=ascentTimeRelative)
    counts_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    weights_all = np.zeros((len(time_bins)-1, len(z_bins)-1))
    
    for p in range(dataDictionary['z'].shape[1]):
        time_relative,z_relative,data, _,_ = PrepareData_Function(dataDictionary, p,
                                                                  varName=varName,relative=relative,ascentTimeRelative=ascentTimeRelative)


        validMask=np.isfinite(data)
        
        c, xedges, yedges = np.histogram2d(time_relative[validMask], z_relative[validMask], bins=[time_bins, z_bins])
        w, _, _ = np.histogram2d(time_relative[validMask], z_relative[validMask], bins=[xedges, yedges], weights=data[validMask])
        counts_all += c
        weights_all += w
    
    meanData = np.where(counts_all > 0, weights_all / counts_all, np.nan)
    return meanData, time_bins,z_bins

def MakeCalculations(PrepareData_Function,arrayDictionary,
                     varNames,parcelTypes,parcelDepth='ALL',relative='LFC',ascentTimeRelative=True):
    meanDataDictionary = {}
    for varName in tqdm(varNames):
        meanDataDictionary[varName] = {}
        for parcelType in parcelTypes:
            dataDictionary = arrayDictionary[parcelType][parcelDepth]
            meanData, time_bins, z_bins = BinData(PrepareData_Function,
                                                  dataDictionary,varName=varName, relative=relative,ascentTimeRelative=ascentTimeRelative)
            meanDataDictionary[varName][parcelType] = meanData    
    return meanDataDictionary, time_bins,z_bins

def CombineMeanData(meanDataDictionary_before, time_bins_before,
                    meanDataDictionary_after,  time_bins_after):
    
    # combined time bins: drop last edge of before (=0) to avoid duplicate
    time_bins_combined = np.concatenate([time_bins_before[:-1], time_bins_after])

    meanDataDictionary_combined = {}
    for varName in meanDataDictionary_before:
        meanDataDictionary_combined[varName] = {}
        for parcelType in meanDataDictionary_before[varName]:
            before = meanDataDictionary_before[varName][parcelType]  # (Nt_before, Nz)
            after  = meanDataDictionary_after[varName][parcelType]   # (Nt_after,  Nz)
            meanDataDictionary_combined[varName][parcelType] = np.concatenate([before, after], axis=0)

    return meanDataDictionary_combined, time_bins_combined

In [ ]:
##############################################
#ASCENT TRAJECTORY HISTOGRAM CALCULATION FUNCTIONS

In [ ]:
# Calculating Functions
# Calculating Functions (Up Until First BL Acceleration)
def PrepareData_UpUntilFirstBLAcceleration(dataDictionary,p=0,varName='W',relative=None,ascentTimeRelative=True,
                                           restrictingToSpecifiedThresholdUpdrafts=False):
    #getting
    z = dataDictionary['z'][:,p]
    qcqi = dataDictionary['QCQI'][:,p]
    LFC = dataDictionary['LFC'][:,p]
    data = dataDictionary[varName][:,p]
    time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

    mask = np.isfinite(z)
    
    #masking
    time = time[mask]; 
    time_relative = time-time[-1] if ascentTimeRelative else time
    z=z[mask]
    data=data[mask]
    z_relative = z #just original z

    #testing ensuring looking at A_g=1 or A_c=1 #*#*#*
    if restrictingToSpecifiedThresholdUpdrafts:
        w = dataDictionary['W'][:,p]
        validMask = ((w[mask]>0.1) & (qcqi[mask]<=1e-6)) | ((w[mask]>0.5) & (qcqi[mask]>1e-6)) #==> A_g==1 or A_c==1
        time_relative = time_relative[validMask]
        z_relative = z_relative[validMask]
        data = data[validMask]
    
    z_bins = np.arange(0, 6000+25, 25)
    if ascentTimeRelative:
        time_bins = np.arange(-30, 0+1, int(ModelData.dt/60))
    else:
        hours=1/(ModelData.dt/3600)
        time_bins = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)[int(4*hours):]*60
    return time_relative,z_relative,data, time_bins,z_bins

# Calculating Functions (First BL Acceleration to Cloud Exit)
def PrepareData_FirstBLAccelerationToCloudExit(dataDictionary,p=0,varName='W',relative='LFC',ascentTimeRelative=True,
                                               restrictingToSpecifiedThresholdUpdrafts=False):
    #getting
    z = dataDictionary['z'][:,p]
    qcqi = dataDictionary['QCQI'][:,p]
    LFC = dataDictionary['LFC'][:,p]
    data = dataDictionary[varName][:,p]
    time = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)*60 

    mask = np.isfinite(z)
    
    #masking
    time = time[mask]; 
    time_relative = time-time[0] if ascentTimeRelative else time
    z=z[mask]
    data=data[mask]
    if relative=='CB':
        cloudMask = qcqi[mask] > 1e-6
        if not np.any(cloudMask):
            time_relative[:] = np.nan
            z_relative = np.full_like(z, np.nan)
            data[:] = np.nan
        else: 
            CB = z[cloudMask][0]
            z_relative = z - CB    
        z_bins = np.arange(-3000, 6000+50, 50)
    elif relative=='LFC':
        LFC=LFC[mask]; z_relative = z-LFC
        z_bins = np.arange(-3000, 6000+50, 50)
    elif relative is None:
        z_relative = z
        z_bins = np.arange(0, 6000+25, 25)

    #testing ensuring looking at A_g=1 or A_c=1 #*#*#*
    if restrictingToSpecifiedThresholdUpdrafts:
        w = dataDictionary['W'][:,p]
        validMask = ((w[mask]>0.1) & (qcqi[mask]<=1e-6)) | ((w[mask]>0.5) & (qcqi[mask]>1e-6)) #==> A_g==1 or A_c==1
        time_relative = time_relative[validMask]
        z_relative = z_relative[validMask]
        data = data[validMask]

        
    if ascentTimeRelative:
        time_bins = np.arange(0, 60+1, int(ModelData.dt/60))
    else:
        hours=1/(ModelData.dt/3600)
        time_bins = (np.arange(ModelData.Ntime)*ModelData.dt/3600+6)[int(4*hours):]*60
    return time_relative,z_relative,data, time_bins,z_bins

# #testing
# p=3
# [time_relative,z_relative,data, time_bins,z_bins] = PrepareData(dataDictionary,p,varName='W')
# plt.plot(time_relative,z_relative)
# plt.ylabel('z - z_LFC (km)'); plt.xlabel('time since ascent (mins)')
# plt.axhline(0, color='grey', zorder=-10, linestyle='--',label='LFC')

In [ ]:
##############################################
#PLOTTING

In [ ]:
##############################################
#PLOTTING (AscentTime-Relative Z-Relative Histograms)

In [ ]:
#Making Calculations and Plotting

if not running and dataName!="ResidenceTime":
    parcelDepth = 'ALL'
    parcelTypesList=[['CL', 'nonCL'],['SBF', 'ColdPool']]
    # parcelTypesList=[parcelTypesList[0]] #*testing
    
    for parcelTypes in parcelTypesList:
        #Calculating
        varNames=GetPlottingVarNames()
        # varNames=['QV','THETA_v','W'] #*testing
        
        [meanDataDictionary_before, time_bins_before,z_bins] = MakeCalculations(PrepareData_UpUntilFirstBLAcceleration,priorToAscentArrayDictionary,
                                                                  varNames,parcelTypes,parcelDepth=parcelDepth,relative=None)
        [meanDataDictionary_after, time_bins_after,z_bins] = MakeCalculations(PrepareData_FirstBLAccelerationToCloudExit,trajectoriesArrayDictionary,
                                                                  varNames,parcelTypes,parcelDepth=parcelDepth,relative=None)
    
        meanDataDictionary_combined, time_bins_combined = CombineMeanData(meanDataDictionary_before, time_bins_before,
                                                                          meanDataDictionary_after,  time_bins_after)

        #Plotting
        plottingFileName = GetPlottingDirectory(plotFileName='AscentTime-Relative_Z-Relative_Histograms', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            for varName in tqdm(varNames):
                
                meanData_CL = meanDataDictionary_combined[varName][parcelTypes[0]]
                meanData_nonCL = meanDataDictionary_combined[varName][parcelTypes[1]]
                
                fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins_combined,z_bins, 
                                     varName=varName,
                                     multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                     cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName),
                                     label=labelDictionary[varName], zLabel='z',titles=parcelTypes,xLabel='time since right before ascent (mins)')
    
                SaveFigureToPDF(pdf,fig,plottingFileName)

In [ ]:
##############################################
#PLOTTING (AscentTime-Relative CB-Relative Histograms)

In [ ]:
#Making Calculations and Plotting

if not running and dataName!="ResidenceTime":
    
    relative='CB' #relative='LFC'
    parcelTypesList=[['CL', 'nonCL'],['SBF', 'ColdPool']]
    # parcelTypesList=[parcelTypesList[0]] #*testing
    
    for parcelTypes in parcelTypesList:
        
        #Calculating
        varNames=GetPlottingVarNames()
        [meanDataDictionary, time_bins,z_bins] = MakeCalculations(PrepareData_FirstBLAccelerationToCloudExit,trajectoriesArrayDictionary,
                                                                  varNames,parcelTypes,parcelDepth=parcelDepth,relative=relative)

        #Plotting
        plottingFileName = GetPlottingDirectory(plotFileName='AscentTime-Relative_CB-Relative_Histograms', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            for varName in tqdm(varNames):
                meanData_CL = meanDataDictionary[varName][parcelTypes[0]]
                meanData_nonCL = meanDataDictionary[varName][parcelTypes[1]]
                fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins,z_bins, 
                                     varName=varName,
                                     multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                     cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName), 
                                     label=labelDictionary[varName], zLabel=f'z - z_{relative} (m)',
                                     titles=parcelTypes,xLabel='time since right before ascent (mins)')
    
                SaveFigureToPDF(pdf,fig,plottingFileName)

In [ ]:
##############################################
#PLOTTING (DomainTime-Relative Z-Relative Histograms After Initial Acceleration)

In [ ]:
#Making Calculations and Plotting

if not running and dataName!="ResidenceTime":
    parcelDepth = 'ALL'
    parcelTypesList=[['CL', 'nonCL'],['SBF', 'ColdPool']]
    parcelTypesList=[parcelTypesList[0]] #*testing
    
    for parcelTypes in parcelTypesList:

        #Calculating
        varNames=GetPlottingVarNames()
        # varNames=['QV','THETA_v'] #*testing
        [meanDataDictionary, time_bins,z_bins] = MakeCalculations(PrepareData_FirstBLAccelerationToCloudExit,trajectoriesArrayDictionary,
                                                                  varNames,parcelTypes,parcelDepth=parcelDepth,
                                                                  relative=None,ascentTimeRelative=False)
        time_bins=time_bins/60

        #Plotting
        plottingFileName = GetPlottingDirectory(plotFileName='DomainTime-Relative_Z-Relative_Histograms', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            for varName in tqdm(varNames):
                meanData_CL = meanDataDictionary[varName][parcelTypes[0]]
                meanData_nonCL = meanDataDictionary[varName][parcelTypes[1]]
                fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins,z_bins, 
                                     varName=varName,
                                     multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                     cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName),
                                     label=labelDictionary[varName], zLabel='z',titles=parcelTypes,xLabel='time (hrs)')
                SaveFigureToPDF(pdf,fig,plottingFileName)

In [ ]:
##############################################
#PLOTTING (DomainTime-Relative CB-Relative Histograms After Initial Acceleration)

In [ ]:
#Making Calculations and Plotting

if not running and dataName!="ResidenceTime":
    parcelDepth = 'ALL'
    parcelTypesList=[['CL', 'nonCL'],['SBF', 'ColdPool']]
    parcelTypesList=[parcelTypesList[0]] #*testing
    
    for parcelTypes in parcelTypesList:
    
        #Calculating
        varNames=GetPlottingVarNames()
        [meanDataDictionary, time_bins,z_bins] = MakeCalculations(PrepareData_FirstBLAccelerationToCloudExit,trajectoriesArrayDictionary,
                                                                  varNames,parcelTypes,parcelDepth=parcelDepth,
                                                                  relative=relative,ascentTimeRelative=False)
        time_bins=time_bins/60
    
        #Plotting
        plottingFileName = GetPlottingDirectory(plotFileName='DomainTime-Relative_CB-Relative_Histograms', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            for varName in tqdm(varNames):
                meanData_CL = meanDataDictionary[varName][parcelTypes[0]]
                meanData_nonCL = meanDataDictionary[varName][parcelTypes[1]]
                fig = PlotBinnedData(meanData_CL,meanData_nonCL,time_bins,z_bins, 
                                     varName=varName,
                                     multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                     cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName), 
                                     label=labelDictionary[varName], zLabel=f'z - z_{relative} (m)',titles=parcelTypes,xLabel='time (hrs)')
                SaveFigureToPDF(pdf,fig,plottingFileName)

In [ ]:
##############################################
#Single-level Distributions Plotting FUNCTIONS

In [ ]:
#Calculation Functions
def CalculateData_BLvsCBvsCEDistribution(dataDictionary,varName):
    BLAscent_Indexes=[]
    BLAscent_Values,CB_Values,LFC_Values=[],[],[]
    BLAscent_Z_Values,CB_Z_Values,LFC_Z_Values,CE_Z_Values,CEminusCB_Z_Values = [],[],[],[],[]
    for p in range(dataDictionary['z'].shape[1]):
        BLAscentIndex = GetTrajectoryBLAscentTime(dataDictionary,p)
        CBIndex = GetTrajectoryCloudBaseTime(dataDictionary,p)
        LFCIndex = GetTrajectoryLFCTime(dataDictionary,p)
        exitIndex = GetTrajectoryLastCloudExitTime(dataDictionary,p)
        
        if CBIndex and LFCIndex:
            BLAscent_Indexes.append(BLAscentIndex)
            varAtBLAscent=dataDictionary[varName][:,p][BLAscentIndex].item()
            varAtCB=dataDictionary[varName][:,p][CBIndex].item()
            varAtLFC=dataDictionary[varName][:,p][LFCIndex].item()
            
            zAtBLAscent=dataDictionary['z'][:,p][BLAscentIndex].item()
            zAtCB=dataDictionary['z'][:,p][CBIndex].item()
            zAtLFC=dataDictionary['z'][:,p][LFCIndex].item()
            zAtCE=dataDictionary['z'][:,p][exitIndex].item()
            zAtCE_MINUS_zAtCB=zAtCE-zAtCB
            
            BLAscent_Values.append(varAtBLAscent)
            CB_Values.append(varAtCB)
            LFC_Values.append(varAtLFC)
            
            BLAscent_Z_Values.append(zAtBLAscent/1e3)
            CB_Z_Values.append(zAtCB/1e3)
            LFC_Z_Values.append(zAtLFC/1e3)
            CE_Z_Values.append(zAtCE/1e3)
            CEminusCB_Z_Values.append(zAtCE_MINUS_zAtCB/1e3)
    return (BLAscent_Indexes,
            BLAscent_Values,CB_Values,LFC_Values,
            BLAscent_Z_Values,CB_Z_Values,LFC_Z_Values,
            CE_Z_Values,CEminusCB_Z_Values)
def GetTrajectoryBLAscentTime(dataDictionary,p):
    array=dataDictionary['z'][:,p]
    BLAscentIndex=FindBLAscentTime(array)
    return BLAscentIndex
def FindBLAscentTime(array):
    BLAscentIndex = np.argmax(~np.isnan(array))+1
    return BLAscentIndex
def GetTrajectoryLastCloudExitTime(dataDictionary,p):
    array=dataDictionary['z'][:,p]
    exitIndex=FindCloudExitTime(array)
    return exitIndex
def GetTrajectoryCloudBaseTime(dataDictionary,p):
    array=dataDictionary['QCQI'][:,p]
    CBIndex=FindCloudBaseTime(array)
    return CBIndex
def GetTrajectoryLFCTime(dataDictionary,p):
    z=dataDictionary['z'][:,p]
    LFC=dataDictionary['LFC'][:,p]
    LFCIndex=FindLFCTime(z,LFC)
    return LFCIndex
def FindCloudExitTime(array):
    exitIndex = len(array) - 1 - np.argmax(~np.isnan(array[::-1])) - 1
    return exitIndex
def FindCloudBaseTime(array):
    CBIndex = np.where(array > 1e-6)[0]
    if len(CBIndex) == 0:
        return []
    return CBIndex[0]
def FindLFCTime(z,LFC):
    LFCIndex = np.where(z >= LFC)[0]
    if len(LFCIndex) == 0:
        return []
    return LFCIndex[0]

In [ ]:
#Calculation Functions

def Calculate_InitialBLAscent_Properties_Histogram(trajectoriesArrayDictionary,varNames,parcelType,parcelDepth='ALL'):  
    times=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    time_bins=np.linspace(10,times[-1],50)
    initialBLAscent_PropertiesVarBinDictionary = GetInitialBLAscent_PropertiesVarBinDictionary()

    dataDictionary=trajectoriesArrayDictionary[parcelType][parcelDepth]
    
    histogramDataDictionary={}
    for varName in tqdm(varNames):
        
        
        [BLAscent_Indexes,
         BLAscent_Values,_,_,
         _,_,_,
         _,_,] = CalculateData_BLvsCBvsCEDistribution(dataDictionary,varName)
        
        timeData=np.array(BLAscent_Indexes)
        timeData=times[BLAscent_Indexes]
            
        varData=np.array(BLAscent_Values)*multiplierDictionary[varName]
        var_bins=initialBLAscent_PropertiesVarBinDictionary[varName]
    
        [counts,time_bins,var_bins]=BinCountHistogram(varData1=timeData,varData2=varData, 
                                                      var1_bins=time_bins,var2_bins=var_bins)
    
        histogramDataDictionary[varName]=counts
    return [histogramDataDictionary, time_bins]

def Calculate_InitialBLAscent_Properties_Histogram_JointDistribution(trajectoriesArrayDictionary,
                                                                     varNames,parcelType,parcelDepth='ALL'):  
    
    initialBLAscent_PropertiesVarBinDictionary = GetInitialBLAscent_PropertiesVarBinDictionary_JointDistribution()

    dataDictionary=trajectoriesArrayDictionary[parcelType][parcelDepth]
    
    jointHistogramDataDictionary={}

    varName1,varName2=varNames
    
    [BLAscent_Indexes1,BLAscent_Values1,
     _,_,_,_,_,_,_,] = CalculateData_BLvsCBvsCEDistribution(dataDictionary,varName1)

    [BLAscent_Indexes2,BLAscent_Values2,
     _,_,_,_,_,_,_,] = CalculateData_BLvsCBvsCEDistribution(dataDictionary,varName2)

    varData1=np.array(BLAscent_Values1)*multiplierDictionary[varName1]
    varData2=np.array(BLAscent_Values2)*multiplierDictionary[varName2]
    var1_bins=initialBLAscent_PropertiesVarBinDictionary[varName1]
    var2_bins=initialBLAscent_PropertiesVarBinDictionary[varName2]

    [counts,_,_]=BinCountHistogram(varData1=varData1,varData2=varData2, 
                                                  var1_bins=var1_bins,var2_bins=var2_bins)

    jointHistogramDataDictionary[f'{varName1}_{varName2}']=counts
    
    return [jointHistogramDataDictionary, var1_bins,var2_bins]

def BinCountHistogram(varData1, varData2, var1_bins, var2_bins):
    validMask = np.isfinite(varData1) & np.isfinite(varData2)
    counts, time_edges, var_edges = np.histogram2d(varData1[validMask], varData2[validMask], bins=[var1_bins, var2_bins])
    return counts, time_edges, var_edges

def GetInitialBLAscent_PropertiesVarBinDictionary():
    
    stepDictionary = {
        "z":                    50/1e3,
        
        "W":                    0.010,
        "VMF_g":                0.010,
        "HMC":                  0.0025,
        
        "QV":                   0.1,
        "THETA_v":              0.1,
        "THETA_e":              0.25,
        "QV_perturbation":      0.1,
        "THETA_v_perturbation": 0.1,
        "THETA_e_perturbation": 0.2,
    }

    # stepDictionary = {k: v / 2 for k, v in stepDictionary.items()} #testing
    
    rangeDictionary = {
        "z":                    (0,1000/1e3),
        
        "W":                    (0.1,   0.3),
        "VMF_g":                (0.1,   0.3),
        "HMC":                  (-0.05,   0.1),
        
        "QV":                   (10,   20),
        "THETA_v":              (302,  312),
        "THETA_e":              (335,  350),
        "QV_perturbation":      (-4,   3),
        "THETA_v_perturbation": (-4,   4),
        "THETA_e_perturbation": (-8,   8),
    }
    
    initialBLAscent_PropertiesVarBinDictionary = {
        varName: np.arange(a, b + step, step)
        for varName, (a, b) in rangeDictionary.items()
        for step in [stepDictionary[varName]]
    }

    return initialBLAscent_PropertiesVarBinDictionary

def GetInitialBLAscent_PropertiesVarBinDictionary_JointDistribution():
    
    stepDictionary = {
        "THETA_v":              0.1,
        "THETA_e":              0.25,
        "QV_perturbation":      0.1,
        "THETA_v_perturbation": 0.1,
        "THETA_e_perturbation": 0.2,
    }
    
    rangeDictionary = {
        "THETA_v":              (304,  312),
        "THETA_e":              (340,  350),
        "QV_perturbation":      (-4,   4),
        "THETA_v_perturbation": (-6,   6),
        "THETA_e_perturbation": (-6,   6),
    }
    
    initialBLAscent_PropertiesVarBinDictionary = {
        varName: np.arange(a, b + step, step)
        for varName, (a, b) in rangeDictionary.items()
        for step in [stepDictionary[varName]]
    }

    return initialBLAscent_PropertiesVarBinDictionary

In [ ]:
#Plotting Functions

def Plot_InitialBLAscent_Properties_Histogram(histogramDataDictionary, time_bins, varNames,
                                              numColumns=3, isNormalize=True, normalizeVMax=20,
                                              plotType = 'contourf'):

    n_vars = len(varNames)
    n_rows = int(np.ceil(n_vars / numColumns))

    fig, axes = plt.subplots(n_rows, numColumns, figsize=(4 * numColumns, 4 * n_rows), sharey=False,
                             gridspec_kw={'hspace': 0.20,'wspace': 0.30})
    axes = np.array(axes).reshape(n_rows, numColumns)

    if isNormalize:
        norm = None
        vmax = normalizeVMax
        colorbarLabel = 'Percent (%)'
        cmap='turbo'
        extend = 'max' if vmax != 100 else None
    else:
        vmax = max(np.nanmax(histogramDataDictionary[v]) for v in varNames)
        norm = mcolors.LogNorm(vmin=1, vmax=vmax)
        colorbarLabel = 'Count'
        cmap='plasma'
        extend = 'max' if vmax != 100 else None

    varBinDictionary = GetInitialBLAscent_PropertiesVarBinDictionary()

    ims = []
    for idx, varName in enumerate(varNames):

        row, col = divmod(idx, numColumns)
        ax = axes[row, col]

        counts = histogramDataDictionary[varName].astype(float)

        if isNormalize:
            counts = 100 * np.divide(counts, np.sum(counts, axis=1, keepdims=True),
                                     out=np.zeros_like(counts), where=np.sum(counts, axis=1, keepdims=True) > 0)

        var_bins = varBinDictionary[varName]
        counts_masked = np.ma.masked_where(counts == 0, counts)

        # im = ax.pcolormesh(time_bins, var_bins, counts_masked.T,
        #                    norm=norm, cmap=cmap, shading='flat',
        #                    vmin=0 if isNormalize else None,
        #                    vmax=vmax if isNormalize else None)

        if plotType == 'pcolormesh':
            im = ax.pcolormesh(time_bins, var_bins, counts_masked.T,
                               norm=norm, cmap=cmap, shading='flat',
                               vmin=0 if isNormalize else None,
                               vmax=vmax if isNormalize else None)
        
        elif plotType == 'contourf':
            levels = np.linspace(0, vmax, 21) if isNormalize else np.geomspace(1, vmax, 21)
        
            im = ax.contourf(time_bins[:-1], var_bins[:-1], counts_masked.T,
                             levels=levels, norm=norm, cmap=cmap,
                             extend=extend)

        ims.append(im)
        label=labelDictionary[varName]
        ax.set_xlabel('time (hr)', fontsize=9)
        ax.set_ylabel(fr'${label}$ ({unitsDictionary[varName]})', fontsize=9)
        ax.tick_params(labelsize=8)

    for idx in range(n_vars, n_rows * numColumns):
        row, col = divmod(idx, numColumns)
        axes[row, col].axis('off')

    fig.subplots_adjust(right=0.88, wspace=0.25, hspace=0.35)
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    fig.colorbar(ims[-1], cax=cax, label=colorbarLabel,extend=extend)

    return fig

# def StackFigs(fig1, fig2, direction='vertical', dpi=150):
#     buf1, buf2 = io.BytesIO(), io.BytesIO()
#     fig1.savefig(buf1, format='png', bbox_inches='tight', dpi=dpi)
#     fig2.savefig(buf2, format='png', bbox_inches='tight', dpi=dpi)
#     buf1.seek(0); buf2.seek(0)
#     img1 = plt.imread(buf1)
#     img2 = plt.imread(buf2)

#     if direction == 'vertical':
#         figsize = (max(img1.shape[1], img2.shape[1]) / dpi,
#                    (img1.shape[0] + img2.shape[0]) / dpi)
#         fig, axes = plt.subplots(2, 1, figsize=figsize,
#                                  gridspec_kw={'height_ratios': [img1.shape[0], img2.shape[0]]})
#     else:  # horizontal
#         figsize = ((img1.shape[1] + img2.shape[1]) / dpi,
#                    max(img1.shape[0], img2.shape[0]) / dpi)
#         fig, axes = plt.subplots(1, 2, figsize=figsize,
#                                  gridspec_kw={'width_ratios': [img1.shape[1], img2.shape[1]]})

#     axes[0].imshow(img1); axes[0].axis('off')
#     axes[1].imshow(img2); axes[1].axis('off')
#     fig.subplots_adjust(hspace=0, wspace=0, left=0, right=1, top=1, bottom=0)
#     return fig

def Plot_InitialBLAscent_Properties_Histogram_JointDistribution(jointHistogramDataDictionary, var1_bins, var2_bins, varNames):

    fig, axis = plt.subplots(figsize=(5, 4))

    varName1,varName2=varNames

    key = f'{varName1}_{varName2}'
    counts = jointHistogramDataDictionary[key]
    countsMasked = np.ma.masked_where(counts == 0, counts)

    vmax = 3e2
    if vmax is None:
        vmax = np.nanmax(countsMasked)
        extend=None
    else:
        extend='max'
    
    norm = mcolors.LogNorm(vmin=1, vmax=vmax)

    im = axis.pcolormesh(
        var1_bins,
        var2_bins,
        countsMasked.T,
        norm=norm,
        cmap='plasma',
        shading='flat'
    )

    label1=labelDictionary[varName1]
    label2=labelDictionary[varName2]
    axis.set_xlabel(fr'${label1}$ ({unitsDictionary[varName1]})')
    axis.set_ylabel(fr'${label2}$ ({unitsDictionary[varName2]})')

    fig.colorbar(im, ax=axis, label='Count', extend=extend)

    
    if '_perturbation' in varNames[0]: 
        axis.axvline(0,color='gray',linestyle='--');axis.axhline(0,color='gray',linestyle='--')

    return fig

In [ ]:
#Plotting Functions

#Figure One
def MakePlot_SingleLevelDistribution(CB_Values_CL, CB_Values_nonCL, varName,parcelTypes, numBins=51,alpha=1,
                                     multiplier=1,units='',atWhichLevel='CB'):
    fig, axis = plt.subplots()

    plotData1=multiplier*np.array(CB_Values_CL)
    plotData2=multiplier*np.array(CB_Values_nonCL)
    axis.hist(plotData1, bins=numBins, color='blue', alpha=alpha, label=parcelTypes[0])
    axis.hist(plotData2, bins=numBins, color='black', alpha=alpha, label=parcelTypes[1])

    axis.set_xlabel(f'{varName} ({units}) at {atWhichLevel}')
    axis.set_ylabel('Count')
    axis.legend()

    return fig

def MakePlot_CBvsLFCScatter(CB_Values_CL, LFC_Values_CL,plotColorData_CL,
                            CB_Values_nonCL, LFC_Values_nonCL,plotColorData_nonCL,
                            varName,parcelTypes,
                            multiplier=1,units='',cmap='turbo',colorbarLabel='z at CB (z)'):

    fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True, sharey=True)

    plotData1=multiplier*np.array(CB_Values_CL); plotData2=multiplier*np.array(LFC_Values_CL); plotColorData=np.array(plotColorData_CL)
    scatter1 = axes[0].scatter(plotData1, plotData2,
                               c=plotColorData, cmap=cmap)

    colorbar1 = fig.colorbar(scatter1, ax=axes[0])
    colorbar1.set_label(colorbarLabel)

    axes[0].set_ylabel(f'{varName} ({units}) at LFC')
    axes[0].set_title(parcelTypes[0])

    plotData1=multiplier*np.array(CB_Values_nonCL); plotData2=multiplier*np.array(LFC_Values_nonCL); plotColorData=np.array(plotColorData_nonCL)
    scatter2 = axes[1].scatter(plotData1, plotData2,
                               c=plotColorData, cmap=cmap)

    colorbar2 = fig.colorbar(scatter2, ax=axes[1])
    colorbar2.set_label(colorbarLabel)

    axes[1].set_xlabel(f'{varName} ({units}) at CB')
    axes[1].set_ylabel(f'{varName} ({units}) at LFC')
    axes[1].set_title(parcelTypes[1])

    fig.tight_layout()

    return fig

In [ ]:
#Plotting Functions

def MakePlot_CBvsCEDistribution_2Panel(CB_Values_CL, CE_Z_Values_CL, plotColorData_CL,
                                       CB_Values_nonCL, CE_Z_Values_nonCL, plotColorData_nonCL,
                                       varName, parcelTypes,
                                       multiplier=1,units='', cmap='turbo',
                                       symmetric=True, numLevels=21,
                                       vmin_threshold=None,colorLabel=None):

    fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharey=True, sharex=True)

    yLabel = 'z_CE - z_CB'

    colorData_CL = multiplier * np.array(plotColorData_CL)
    colorData_nonCL = multiplier * np.array(plotColorData_nonCL)
    allColorData = np.concatenate([colorData_CL, colorData_nonCL])

    if vmin_threshold is not None:
        vmin_threshold *= multiplier
        colorData_CL = np.where(colorData_CL >= vmin_threshold, colorData_CL, np.nan)
        colorData_nonCL = np.where(colorData_nonCL >= vmin_threshold, colorData_nonCL, np.nan)

    if symmetric:
        vmax = np.nanpercentile(np.abs(allColorData), 95)
        vmin = -vmax
        # levels = np.linspace(vmin, vmax, numLevels + 1)
        levelsNeg = np.linspace(vmin, 0, numLevels//2 + 1); levelsPos = np.linspace(0, vmax, numLevels//2 + 1)
        levels = np.concatenate([levelsNeg[:-1], levelsPos])
        # norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        norm = mcolors.BoundaryNorm(boundaries=levels,ncolors=plt.get_cmap(cmap).N,clip=True)
        extend = 'both'
    else:
        vmin = np.nanpercentile(allColorData, 5)
        vmax = np.nanpercentile(allColorData, 95)
        if vmax == 0 and '_E_' in varName or '_D_' in varName:
            vmax = np.nanmax(allColorData)
        levels = np.linspace(vmin, vmax, numLevels + 1)
        #norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        norm = mcolors.BoundaryNorm(boundaries=levels,ncolors=plt.get_cmap(cmap).N,clip=True)
        extend = 'both'

    # --- CL ---
    plotData1 = multiplier * np.array(CB_Values_CL)
    plotData2 = np.array(CE_Z_Values_CL)

    scatterPlot1 = axes[0].scatter(plotData1, plotData2,
                                   c=colorData_CL,cmap=cmap,norm=norm)

    axes[0].set_ylabel(yLabel); axes[0].set_title(parcelTypes[0])

    # --- nonCL ---
    plotData1 = multiplier * np.array(CB_Values_nonCL)
    plotData2 = np.array(CE_Z_Values_nonCL)

    scatterPlot2 = axes[1].scatter(plotData1, plotData2,c=colorData_nonCL,cmap=cmap,norm=norm)

    label=labelDictionary[varName]
    axes[1].set_xlabel(f"${label}$ ({units}) at CB"); axes[1].set_ylabel(yLabel); axes[1].set_title(parcelTypes[1])

    colorbar = fig.colorbar(scatterPlot2,ax=axes,ticks=levels,extend=extend)
    if colorLabel is not None: colorbar.set_label(colorLabel)

    #adjusting xTicks
    varBins = GetVarBinDictionary()[varName]
    for axis in axes:
        axis.set_xlim(varBins[0],varBins[-1])
    #adjusting yTicks
    zBins = GetVarBinDictionary()['z']
    for axis in axes:
        axis.set_ylim(zBins[0],zBins[-1])

    # fig.tight_layout()
    return fig

def MakePlot_CBvsCEDistribution_2DHistogram(CB_Values_CL,Z_Values_CL,
                                       CB_Values_nonCL,Z_Values_nonCL,
                                       varName,parcelTypes,
                                            multiplier=1,units='',cmap='turbo',useLog=False,
                                       plotType='pcolormesh'):

    fig, axes = plt.subplots(2, 1,figsize=(7, 8),sharex=True,sharey=True)

    varBinDictionary = GetVarBinDictionary()
    zBins = varBinDictionary['z']
    varBins = varBinDictionary[varName]

    yLabel = 'z_CE - z_CB'

    plotData1=multiplier*np.array(CB_Values_CL); plotData2=np.array(Z_Values_CL) 
    histCL, xEdges, yEdges = np.histogram2d(plotData1,plotData2,bins=[varBins, zBins])

    plotData1=multiplier*np.array(CB_Values_nonCL); plotData2=np.array(Z_Values_nonCL) 
    histNonCL, _, _ = np.histogram2d(plotData1,plotData2,bins=[varBins, zBins])

    maxCount = max(np.nanmax(histCL), np.nanmax(histNonCL))
    if useLog:
        norm = LogNorm(vmin=1, vmax=maxCount) if maxCount>0 else None
    else:
        norm = None

    xCenters = (xEdges[:-1] + xEdges[1:]) / 2
    yCenters = (yEdges[:-1] + yEdges[1:]) / 2

    def drawPlot(ax, hist):
        if plotType == 'pcolormesh':
            return ax.pcolormesh(xEdges,yEdges,hist.T,cmap=cmap,shading='auto',norm=norm)
        elif plotType == 'contourf':
            hist_masked = np.where(hist > 0, hist, np.nan)
            if useLog and maxCount > 0:
                plot_levels = np.geomspace(1, maxCount, 30)
                tick_levels = LogLocator(base=10).tick_values(1, maxCount)
                tick_levels = tick_levels[(tick_levels >= 1) & (tick_levels <= maxCount)]
            else:
                plot_levels = np.linspace(0, maxCount, 11)
                tick_levels = plot_levels
            ax._contourf_tick_levels = tick_levels
            return ax.contourf(xCenters,yCenters,hist_masked.T,levels=plot_levels,cmap=cmap,norm=norm)
        else:
            raise ValueError(f"Unknown plotType '{plotType}'. Choose 'pcolormesh' or 'contourf'.")

    plot1 = drawPlot(axes[0], histCL)
    colorbar1 = fig.colorbar(plot1, ax=axes[0], ticks=axes[0]._contourf_tick_levels if plotType=='contourf' else None)
    if plotType=='contourf' and useLog: colorbar1.ax.set_yscale('log')
    colorbar1.set_label('Counts')
    axes[0].set_ylabel(yLabel); axes[0].set_title(parcelTypes[0]);

    plot2 = drawPlot(axes[1], histNonCL)
    colorbar2 = fig.colorbar(plot2, ax=axes[1], ticks=axes[1]._contourf_tick_levels if plotType=='contourf' else None)
    if plotType=='contourf' and useLog: colorbar2.ax.set_yscale('log')
    colorbar2.set_label('Counts')
    label=labelDictionary[varName]
    axes[1].set_xlabel(fr'${label}$ ({units}) at CB'); axes[1].set_ylabel(yLabel); axes[1].set_title(parcelTypes[1])
    return fig

#Histogram Bin Dictionaries
def GetVarBinDictionary(numCount=75):
    #Variables
    varBinDictionary = {'QV':        np.linspace(8, 18, numCount),
                        'QV_perturbation':        np.linspace(1.5, 4.5, numCount),
                        'RH_vapor':  np.linspace(100, 101.5, numCount),
                        'RH_vapor_perturbation':  np.linspace(10, 40, numCount),
                        'QCQI':      np.linspace(0, 3, numCount),
                        'THETA_v':   np.linspace(304, 318, numCount),
                        'THETA_v_perturbation':   np.linspace(-2, 2, numCount),
                        'THETA_e':   np.linspace(338, 348, numCount),
                        'THETA_e_perturbation':   np.linspace(2, 14, numCount),
                        'W':         np.linspace(0, 14, numCount),
                        'VMF_c':     np.linspace(0, 10, numCount), 
                        'HMC': np.linspace(-0.06, 0.08, numCount),
                        'BUOYANCY2': np.linspace(-0.06, 0.06, numCount),
                        'z':         np.linspace(0, 12, numCount)}

    # Entrainment
    varBinDictionary.update({'PROCESSED_E_c':                np.linspace(0, 0.03, numCount),
                             'PROCESSED_D_c':                np.linspace(0, 0.03, numCount),
                             'PROCESSED_NET_c':                np.linspace(-0.03, 0.03, numCount),
                             'PROCESSED_E_DivideMassFlux_c': np.linspace(0, 10, numCount),
                             'PROCESSED_D_DivideMassFlux_c': np.linspace(0, 10, numCount),
                             'PROCESSED_NET_DivideMassFlux_c': np.linspace(-10, 10, numCount)})

    # UpdraftArea
    varBinDictionary.update({'UPDRAFT_AREA_c': np.linspace(0, 20, numCount),
                             'UPDRAFT_EDGE_DISTANCE_c': np.linspace(1, 3.5, numCount)})
    return varBinDictionary

In [ ]:
#Plotting Function

def PlotCBDistributions(trajectoriesArrayDictionary, varName,
                        whichFigure='One', relative='CB',parcelTypes=['CL','nonCL'],parcelDepth='ALL'):
    

    #############################
    #ParcelType One
    dataDictionary=trajectoriesArrayDictionary[parcelTypes[0]][parcelDepth]
    [BLAscent_Indexes_CL,
     BLAscent_Values_CL,CB_Values_CL,LFC_Values_CL,
      BLAscent_Z_Values_CL,CB_Z_Values_CL,LFC_Z_Values_CL,
     CE_Z_Values_CL,CEminusCB_Z_Values_CL] = CalculateData_BLvsCBvsCEDistribution(dataDictionary,varName)

    #ParcelType Two
    dataDictionary=trajectoriesArrayDictionary[parcelTypes[1]][parcelDepth]
    [BLAscent_Indexes_nonCL,
     BLAscent_Value_nonCL,CB_Values_nonCL,LFC_Values_nonCL,
      BLAscent_Z_ValuesnonCL,CB_Z_Values_nonCL,LFC_Z_Values_nonCL,
     CE_Z_Values_nonCL,CEminusCB_Z_Values_nonCL] = CalculateData_BLvsCBvsCEDistribution(dataDictionary,varName)
    
    #Figure One
    if whichFigure == "One":
        relative = 'CB'   # or 'LFC'
        values_CL, values_nonCL = {'CB':  (CB_Values_CL,  CB_Values_nonCL),'LFC': (LFC_Values_CL, LFC_Values_nonCL)}[relative]
        zValues_CL, zValues_nonCL = {'CB':  (CB_Z_Values_CL,  CB_Z_Values_nonCL),'LFC': (LFC_Z_Values_CL, LFC_Z_Values_nonCL),}[relative]

        fig1 = MakePlot_SingleLevelDistribution(values_CL,values_nonCL,varName,parcelTypes,
                                         multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                         atWhichLevel=relative)
        fig2 = MakePlot_CBvsLFCScatter(CB_Values_CL, LFC_Values_CL, zValues_CL,
                                CB_Values_nonCL, LFC_Values_nonCL, zValues_nonCL,
                                varName,parcelTypes,
                                multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                colorbarLabel=f'z at {relative} (z)')

        
    #Figure Two
    if whichFigure == "Two":
        fig1 = MakePlot_CBvsCEDistribution_2Panel(CB_Values_CL,CEminusCB_Z_Values_CL,LFC_Values_CL,
                                                 CB_Values_nonCL,CEminusCB_Z_Values_nonCL,LFC_Values_nonCL,
                                                 varName,parcelTypes,
                                                 multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                                 cmap=cmapDictionary[varName],symmetric=isVariableSymmetric(varName),
                                                 colorLabel=fr'${labelDictionary[varName]}$ at LFC')
        fig2 = MakePlot_CBvsCEDistribution_2DHistogram(CB_Values_CL,CEminusCB_Z_Values_CL,
                                                      CB_Values_nonCL,CEminusCB_Z_Values_nonCL, 
                                                      varName,parcelTypes,
                                                      multiplier=multiplierDictionary[varName],units=unitsDictionary[varName],
                                                      useLog=True)

    return fig1,fig2

In [ ]:
#Plotting Functions

def Plot_BLAscent_X(trajectoriesArrayDictionary, parcelTypes=['CL', 'nonCL'],
                    parcelDepth='ALL', plotType='histogram'):

    varNames = ['X', 'X_SBF_Relative']

    xLabelDictionary = {'X': 'X distance (kms)',
                        'X_SBF_Relative': 'X distance from SBF (kms)'}

    if plotType=='histogram':
        limitsDictionary = {'X': (ModelData.Nxh*0.25*ModelData.kms, 450),
                            'X_SBF_Relative': (-100, 250)}
    elif plotType=='timeseries':
        limitsDictionary = {'X': (ModelData.Nxh*0.25*ModelData.kms, 400),
                            'X_SBF_Relative': (-50, 150)}

    numBins = 100

    fig, axes = plt.subplots(len(parcelTypes), len(varNames), figsize=(10, 6), sharey=False)

    for i, parcelType in enumerate(parcelTypes):
        for j, varName in enumerate(varNames):

            axis = axes[i, j]
            data = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]

            if plotType == 'histogram':
                plotData = data[~np.isnan(data)]
                axis.hist(plotData, bins=numBins, color='green', edgecolor='k')
                axis.set_xlabel(xLabelDictionary[varName])
                axis.set_ylabel('Count')
                axis.set_xlim(limitsDictionary[varName])

            elif plotType == 'timeseries':
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    plotData = np.nanmean(data, axis=1)
                axis.plot(ModelData.time_hrs, plotData, color='green')
                axis.set_xlabel('Time (hrs)')
                axis.set_ylabel(xLabelDictionary[varName])
                axis.set_ylim(limitsDictionary[varName])

                axis.axhline(0,color='gray',linestyle='dashed',zorder=-10)

            mu = np.nanmean(data)

            axis.set_title(f'{parcelType} - {parcelDepth}\n$\\mu$ = {mu:.2f}')
            SnapLimitsToTicks(axis, dim='y')

    fig.tight_layout()

    return fig

In [ ]:
##############################################
#PLOTTING (Single-level Distributions)

In [ ]:
#Plotting
if not running and dataName=='Variables':

    parcelDepth = 'ALL'
    for parcelTypes in [['CL', 'nonCL'],['SBF', 'ColdPool']]:
        
        
        varNames = ['W','VMF_g','HMC',
                    'QV','THETA_v','THETA_e',
                    'QV_perturbation','THETA_v_perturbation','THETA_e_perturbation'] #'z'
    
        plottingFileName = GetPlottingDirectory(plotFileName=f'InitialBLAscent_Properties_Histogram', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            for parcelType in parcelTypes:
                [histogramDataDictionary, time_bins]\
                = Calculate_InitialBLAscent_Properties_Histogram(trajectoriesArrayDictionary,
                                                                 varNames,parcelType=parcelType,parcelDepth=parcelDepth)
                fig=Plot_InitialBLAscent_Properties_Histogram(histogramDataDictionary,
                                                      time_bins,varNames,numColumns=3, isNormalize=True,normalizeVMax=20)
    
                SaveFigureToPDF(pdf,fig,plottingFileName)

In [ ]:
#Plotting
if not running and dataName=='Variables':
    parcelDepth = 'ALL'
    for parcelTypes in [['CL', 'nonCL'],['SBF', 'ColdPool']]:

        
        varNames1=['THETA_v','THETA_e']
        varNames2=['THETA_v_perturbation','THETA_e_perturbation']
        
        varNamesList=[varNames1,varNames2]
        plottingFileName = GetPlottingDirectory(plotFileName=f'InitialBLAscent_Properties_Histogram_JointDistribution', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            for parcelType in parcelTypes: 
                for varNames in varNamesList:
                    
                    [jointHistogramDataDictionary, var1_bins,var2_bins]=\
                    Calculate_InitialBLAscent_Properties_Histogram_JointDistribution(trajectoriesArrayDictionary,
                                                                                     varNames,parcelType,parcelDepth='ALL')
                    
                    fig=Plot_InitialBLAscent_Properties_Histogram_JointDistribution(jointHistogramDataDictionary, 
                                                                                    var1_bins, var2_bins, varNames)
        
                    SaveFigureToPDF(pdf,fig,plottingFileName)

In [ ]:
#Plotting
if not running and dataName!="ResidenceTime":
    parcelDepth = 'ALL'

    parcelTypesList=[['CL', 'nonCL'],['SBF', 'ColdPool']]
    parcelTypesList=[parcelTypesList[0]] #*testing
    
    for parcelTypes in parcelTypesList:
        
        whichFigure='Two'
        plottingFileName = GetPlottingDirectory(plotFileName=f'CB_Distributions_{whichFigure}', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            varNames=GetPlottingVarNames(); varNames.append('z'); varNames=[v for v in varNames if '_g' not in v]; 
            
            for varName in tqdm(varNames):
                [fig1,fig2]=PlotCBDistributions(trajectoriesArrayDictionary,varName,
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth,whichFigure=whichFigure)
        
                SaveFigureToPDF(pdf,fig1,plottingFileName)
                SaveFigureToPDF(pdf,fig2,plottingFileName)

In [ ]:
#Plotting
if not running and dataName=='Variables':
    parcelDepth = 'ALL'
    for parcelTypes in [['CL', 'nonCL'],['SBF', 'ColdPool']]:
       
        plottingFileName = GetPlottingDirectory(plotFileName=f'InitialBLAscent_X_and_XfromSBF', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:      
            for plotType in ['histogram','timeseries']:
                fig = Plot_BLAscent_X(trajectoriesArrayDictionary,parcelTypes=parcelTypes,parcelDepth='ALL', plotType=plotType)
                SaveFigureToPDF(pdf,fig,plottingFileName)

In [ ]:
##############################################
#Comparing Residence Times

In [ ]:
##############################################
#FUNCTIONS

In [ ]:
#CALCULATING
def CalculateMaxes(parcelType,parcelDepth='ALL',residenceTimeType='since_ascent'): 
    
    dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth] 

    time_since=dataDictionary[f'time_{residenceTimeType}']
    z=dataDictionary[f'z_{residenceTimeType}']
    W=dataDictionary[f'W_{residenceTimeType}']
    
    duration = np.nanmax(time_since, axis=0) 
    
    depth = np.nanmax(z, axis=0) / 1e3 # km 
    maxW = np.nanmax(W, axis=0)
    
    return depth, duration, maxW

def CalculateWLagAutocorrelation(parcelType,parcelDepth='ALL', residenceTimeType='since_ascent', lagSteps=1):

    dataDictionary = trajectoriesArrayDictionary[parcelType][parcelDepth]
    W = dataDictionary[f'W_{residenceTimeType}']

    W1 = W[:-lagSteps, :]
    W2 = W[lagSteps:, :]

    valid = np.isfinite(W1) & np.isfinite(W2)

    autocorr = np.full(W.shape[1], np.nan)

    for p in range(W.shape[1]):

        if np.sum(valid[:, p]) > 3:
            autocorr[p] = np.corrcoef(W1[valid[:, p], p],
                                      W2[valid[:, p], p])[0, 1]

    return autocorr

#PLOTTING

def MakePlot_1DHistogram(durationCL, durationNonCL,
                         parcelTypes,
                         residenceTimeType='since_ascent',
                         bins=25, normalize=True,
                         color_CL='purple', color_nonCL='orange',
                         linewidth=1.8,alpha=0.8):

    fig, axis = plt.subplots(figsize=(8, 5))

    duration_max = max(np.nanmax(durationCL), np.nanmax(durationNonCL))
    # duration_bins = np.linspace(0, duration_max, bins) #creates some binning artifacts in 2dHistogram
    mins_per_timesteps=int(1/ModelData.mins)
    duration_bins = np.arange(0, duration_max + mins_per_timesteps, mins_per_timesteps) 
    

    for duration, label, color in zip([durationCL, durationNonCL],
                                      parcelTypes,
                                      [color_CL, color_nonCL]):

        valid = duration[np.isfinite(duration)]
        n = len(valid)
        weights = np.full(n, 100.0 / n) if normalize else None

        axis.hist(valid, bins=duration_bins,
                histtype='step', linewidth=linewidth,
                color=color, alpha=alpha, weights=weights, label=label)

        median = np.nanmedian(valid)
        axis.axvline(median, color=color, linestyle='--', linewidth=1.5,
                   label=f'{label} median ({median:.1f} min)')

    initialLocationString = ("BL Ascent" if residenceTimeType == "since_ascent"
                         else "CB" if residenceTimeType == "since_CB"
                         else "Cloudy Updraft" if residenceTimeType == "since_CU"
                         else "")
    axis.set_xlabel(f'Duration since {initialLocationString} (min)')
    axis.set_ylabel('Percent of parcels (%)' if normalize else 'Count')

    axis.set_title(f'Duration from {initialLocationString} to Cloudy Updraft Exit')
    axis.legend()
    fig.tight_layout()

    return fig


from matplotlib.colors import LogNorm
def MakePlot_2DHistogram(depthCL, durationCL, zCL,
                         depthNonCL, durationNonCL, zNonCL,
                         parcelTypes,
                         cmap='turbo',
                         residenceTimeType='since_ascent',
                         contourLevels=np.arange(2, 32, 2),
                         contourLabel='Maximum W',
                         fmt="%d",
                         vmin=1e-3):

    depth_bins = np.linspace(0, max(np.nanmax(depthCL), np.nanmax(depthNonCL)), 50)
    # duration_bins = np.linspace(0, max(np.nanmax(durationCL), np.nanmax(durationNonCL)), 20) #creates some weird binning artifacts
    mins_per_timesteps=int(1/ModelData.mins)
    duration_bins = np.arange(0, max(np.nanmax(durationCL), np.nanmax(durationNonCL)) + mins_per_timesteps, mins_per_timesteps)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True, gridspec_kw={'wspace':0.03})

    percentList = []

    for depth, duration in zip([depthCL, depthNonCL],
                               [durationCL, durationNonCL]):

        valid = np.isfinite(depth) & np.isfinite(duration)

        counts, _, _ = np.histogram2d(depth[valid], duration[valid],
                                      bins=[depth_bins, duration_bins])

        percent = 100 * counts / np.nansum(counts)
        percentList.append(np.ma.masked_where(percent == 0, percent))

    vmax = max(np.nanmax(percentList[0]), np.nanmax(percentList[1]))
    norm = LogNorm(vmin=vmin, vmax=vmax)

    for axis, depth, duration, zData, percent, title in zip(axes,
                                                            [depthCL, depthNonCL],
                                                            [durationCL, durationNonCL],
                                                            [zCL, zNonCL],
                                                            percentList,
                                                            parcelTypes):

        im = axis.pcolormesh(depth_bins, duration_bins, percent.T,
                             cmap=cmap, norm=norm, shading='flat')

        CalculateAndAdd2DContours(axis, depth, duration, zData,
                                  depth_bins, duration_bins,
                                  levels=contourLevels, colors='magenta', fmt=fmt)

        axis.set_title(title)
        axis.set_xlabel('Max depth reached (km)')

    initialLocationString = ("BL Ascent" if residenceTimeType == "since_ascent"
                         else "CB" if residenceTimeType == "since_CB"
                         else "Cloudy Updraft" if residenceTimeType == "since_CU"
                         else "")
    axes[0].set_ylabel(f'Duration since {initialLocationString} (min)')

    fig.colorbar(im, ax=axes, label='Percent of parcels (%)', pad=0.01,fraction=0.05)

    fig.suptitle(
        f'Maximum Duration vs Depth from {initialLocationString} to Cloudy Updraft Exit (colors)\n'
        f'{contourLabel} (magenta lines)'
    )
    return fig
    
from scipy.stats import binned_statistic_2d
def CalculateAndAdd2DContours(axis, xData, yData, zData, xBins, yBins,
                              levels=np.arange(2, 32, 2),colors='magenta',statistic='mean',fmt='%d',
                              zorder=10):

    valid = np.isfinite(xData) & np.isfinite(yData) & np.isfinite(zData)

    stat, xEdge, yEdge, _ = binned_statistic_2d(xData[valid],yData[valid],zData[valid],
                                                statistic=statistic,bins=[xBins, yBins])

    xCenter = 0.5 * (xEdge[:-1] + xEdge[1:]); yCenter = 0.5 * (yEdge[:-1] + yEdge[1:])

    contour = axis.contour(xCenter,yCenter,
                           stat.T,
                           levels=levels,colors=colors,zorder=zorder)
    axis.clabel(contour, fmt=fmt)
    return contour

In [ ]:
##############################################
#RUNNING

In [ ]:
if not running and dataName=='ResidenceTime':

    parcelDepth = 'ALL'
    parcelTypesList = [['CL', 'nonCL'],['SBF', 'ColdPool']]
    # parcelTypesList=[parcelTypesList[1]] #*testing
    
    for parcelTypes in parcelTypesList:

        plottingFileName = GetPlottingDirectory(plotFileName='Comparing_Residence_Times', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)

        with PdfPages(plottingFileName) as pdf:
            for residenceTimeType in ['since_ascent','since_CU']:
                
                #CALCULATING
                [depthCL, durationCL, maxW_CL] = CalculateMaxes(parcelTypes[0],parcelDepth='ALL',
                                                                residenceTimeType=residenceTimeType)
                [depthNonCL, durationNonCL, maxW_NonCL] = CalculateMaxes(parcelTypes[1],parcelDepth='ALL',
                                                                         residenceTimeType=residenceTimeType)
                
                # wAutoCorr_CL = CalculateWLagAutocorrelation(parcelTypes[0], residenceTimeType=residenceTimeType, lagSteps=1)
                # wAutoCorr_NonCL = CalculateWLagAutocorrelation(parcelTypes[1], residenceTimeType=residenceTimeType, lagSteps=1)
                
                #PLOTTING
                fig1=MakePlot_2DHistogram(depthCL, durationCL, maxW_CL, #wAutoCorr_CL
                                          depthNonCL, durationNonCL, maxW_NonCL, #wAutoCorr_NonCL
                                          parcelTypes,
                                          residenceTimeType=residenceTimeType, #contourLevels=np.arange(0, 1.05, 0.1),
                                          contourLabel='Maximum W') #contourLabel=f"W autocorrelation (lag - {int(ModelData.dt/60)}mins)",fmt='%.1f'

                fig2 = MakePlot_1DHistogram(durationCL, durationNonCL,
                                            parcelTypes,
                                            residenceTimeType=residenceTimeType)
        
                SaveFigureToPDF(pdf,fig1,plottingFileName)
                SaveFigureToPDF(pdf,fig2,plottingFileName)

In [ ]:
##############################################
#Quantify Parcels Previously in Downdrafts

In [ ]:
##############################################
#CALCULATION and PLOTTING FUNCTIONS

In [ ]:
#Calculation Functions

from collections import defaultdict

def CalculateDowndraftStatistics(parcelType,
                                 numPriorMinutes=10,threshold=-0.1):

    allAscentTimes = []; downdraftAscentTimes = []; downdraftStrengths = []
    
    priorToAscentArray = priorToAscentArrayDictionary[parcelType]['ALL']
    times = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
    mins = 1 / (ModelData.dt / 60); priorMinutes = int(numPriorMinutes * mins)
    
    for p in tqdm(range(priorToAscentArray['z'].shape[1])):
        array = priorToAscentArray['W'][:, p]
        lastValue = np.argmax(np.isnan(array)) - 1
        ascentTime = times[lastValue + 1]
        window = array[lastValue - priorMinutes + 1 : lastValue + 1]
    
        # All parcels
        allAscentTimes.append(ascentTime)
    
        # Downdraft parcels
        if np.any(window < threshold):
            downdraftAscentTimes.append(ascentTime)
            downdraftStrengths.append(window.min())
    
    return (np.array(allAscentTimes),
            np.array(downdraftAscentTimes),
            np.array(downdraftStrengths))

#Plotting Functions
def MakeDowndraftStatisticsPlot(allAscentTimes,downdraftAscentTimes,downdraftStrengths,numPriorMinutes):
    
    bins = np.arange(6, 18.5, 0.5)
    binCenters = 0.5 * (bins[:-1] + bins[1:])
    
    allCounts, _ = np.histogram(allAscentTimes, bins=bins)
    downdraftCounts, _ = np.histogram(downdraftAscentTimes, bins=bins)
    fraction = np.where(allCounts > 0, downdraftCounts / allCounts * 100, np.nan)
    
    # Mean strength per bin
    binIndices = np.digitize(downdraftAscentTimes, bins) - 1
    meanStrength = np.full(len(bins) - 1, np.nan)
    for i in range(len(bins) - 1):
        vals = [s for s, b in zip(downdraftStrengths, binIndices) if b == i]
        if vals:
            meanStrength[i] = np.mean(vals)
    
    # Dual-axis plot
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    ax1.plot(binCenters, fraction, color='darkblue', marker='o', label='% in Downdraft')
    ax1.set_xlabel('Ascent Time (hrs)')
    ax1.set_ylabel(f'% Parcels in Downdraft', color='darkblue')
    ax1.tick_params(axis='y', labelcolor='darkblue')
    ax1.set_ylim(0, 100);ax1.set_xlim(10, 17)
    
    ax2 = ax1.twinx()
    ax2.plot(binCenters, meanStrength, color='darkgreen', marker='s', linestyle='--', label='Mean Min W')
    ax2.set_ylabel(f'Minimum W (m/s)', color='darkgreen')
    ax2.tick_params(axis='y', labelcolor='darkgreen')
    ax2.invert_yaxis()
    ax2.set_ylim(bottom=-0.1,top=-2)
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    plt.title(f'Average Statistics of Parcels in Downdraft (w < -0.1) in last {numPriorMinutes} minutes ({parcelType})', color='black')
    plt.tight_layout()

    return fig

# def MakeDowndraftStrengthHistogramPlot(allAscentTimes,downdraftAscentTimes,downdraftStrengths,numPriorMinutes,
#                                        plotType="pcolormesh",cmap='turbo'):
#     bins = np.arange(6, 18.5, 0.5)
#     strengthBins = np.linspace(-3, -0.1, 100)
    
#     # 2D histogram: ascent time vs downdraft strength
#     H, xedges, yedges = np.histogram2d(
#         downdraftAscentTimes,
#         downdraftStrengths,
#         bins=[bins, strengthBins]
#     )
    
#     # Normalize each time column by total parcels at that time
#     allCounts, _ = np.histogram(allAscentTimes, bins=bins)
#     H_norm = np.where(allCounts[:, None] > 0, H / allCounts[:, None] * 100, np.nan)
    
#     fig, ax = plt.subplots(figsize=(12, 5))
    
#     norm = LogNorm(vmin=0.01, vmax=np.nanpercentile(H_norm[H_norm > 0], 99))
#     if plotType == 'pcolormesh':
#         im = ax.pcolormesh(xedges, yedges, H_norm.T, cmap=cmap, shading='flat', norm=norm)
#     elif plotType == 'contourf':
#         xcenters = 0.5 * (xedges[:-1] + xedges[1:])
#         ycenters = 0.5 * (yedges[:-1] + yedges[1:])
#         levels = np.logspace(np.log10(0.01), np.log10(np.nanpercentile(H_norm[H_norm > 0], 99)), 20)
#         im = ax.contourf(xcenters, ycenters, H_norm.T, levels=levels, cmap=cmap, norm=norm)
#     plt.colorbar(im, ax=ax, label='% of All Parcels Ascending at That Time')
    
#     ax.set_xlabel('Ascent Time (hrs)')
#     ax.set_ylabel('Downdraft Strength (m/s)')
#     plt.title(f'Average Statistics of Parcels in Downdraft (w < -0.1) in last {numPriorMinutes} minutes ({parcelType})', color='black')
#     ax.axhline(-0.1, color='dodgerblue', linestyle='--', linewidth=1, label='W = -0.1 threshold')
#     ax.legend()
    
#     plt.tight_layout()
#     plt.xlim(10,17);plt.ylim(top=0)

#     return fig

In [ ]:
##############################################
#PLOTTING (Quantify Parcels Previously in Downdrafts)

In [ ]:
#Plotting
if not running and dataName=='Variables':
    parcelDepth = 'ALL'
    for parcelTypes in [['CL', 'nonCL'],['SBF', 'ColdPool']]:
        
        plottingFileName = GetPlottingDirectory(plotFileName='Quantify_Parcels_Previously_in_Downdrafts', 
                                                parcelTypes=parcelTypes,parcelDepth=parcelDepth)
        with PdfPages(plottingFileName) as pdf:
            for numPriorMinutes in [5,10,20]:
                for parcelType in parcelTypes:
                    [allAscentTimes,downdraftAscentTimes,downdraftStrengths]=\
                    CalculateDowndraftStatistics(parcelType=parcelType,numPriorMinutes=numPriorMinutes)
                    
                    fig=MakeDowndraftStatisticsPlot(allAscentTimes,downdraftAscentTimes,downdraftStrengths,numPriorMinutes)
                    # fig=MakeDowndraftStrengthHistogramPlot(allAscentTimes,downdraftAscentTimes,downdraftStrengths,numPriorMinutes)
                    
                    SaveFigureToPDF(pdf,fig,plottingFileName)